# 13 — V2 grounding benchmark: 24 cases
Run in VS Code with your Agent1 `.venv` kernel. Put the final Colab `adapter_last` contents directly in `models/adapter_v2/`.
Restart any previous notebook kernel holding a model, then leave it idle to free GPU memory. This notebook loads one model and the V2 adapter, without training.

It reuses the original base run's saved prompts, model revision (when recorded), chat template and generation settings. It does not rerun the base model. Answers are saved after each case.

After the run, share `responses.jsonl` and `comparison.json` from the printed results folder. We will review correctness and regressions, then proceed to Agent1 integration. No checkpoint sweep or new training is needed.

**Folder organization:** training logs and previews stay directly in `experiments/grounding_v2/run_20261024T220321_655458Z/`; the 24-case benchmark lives in its `benchmark/` subfolder. The original base results remain in `experiments/grounding/base_.../`. Notebook 12 needs no path change.

In [ ]:
from pathlib import Path
import json, hashlib, sys
import importlib.metadata as metadata
from datetime import datetime, timezone
from time import perf_counter

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "agent1").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Run this notebook from Agent1 or its notebooks folder.")
adapter_dir = project_root / "models" / "adapter_v2"
for name in ["adapter_config.json", "adapter_model.safetensors"]:
    if not (adapter_dir / name).is_file():
        raise FileNotFoundError(f"Copy the final adapter_last contents to {adapter_dir}; missing {name}")

# Keep all V2 training and benchmark results together.
V2_TRAINING_RUN = "run_20261024T220321_655458Z"
v2_run_dir = project_root / "experiments" / "grounding_v2" / V2_TRAINING_RUN

# Usually leave None. If several base runs exist, enter the desired folder name.
BASE_RUN_NAME = None  # Example: "base_20261023T..."
grounding_dir = project_root / "experiments" / "grounding"
if BASE_RUN_NAME:
    base_run_dir = grounding_dir / BASE_RUN_NAME
else:
    candidates = []
    for path in sorted(grounding_dir.glob("base_*/manifest.json")):
        response_path = path.parent / "responses.jsonl"
        if response_path.is_file():
            rows = [json.loads(x) for x in response_path.read_text(encoding="utf-8-sig").splitlines() if x.strip()]
            if len(rows) == 24 and len({r["id"] for r in rows}) == 24:
                candidates.append(path.parent)
    if len(candidates) != 1:
        raise ValueError("Set BASE_RUN_NAME to the completed notebook-09 run folder. Found: "
                         + str([x.name for x in candidates]))
    base_run_dir = candidates[0]
base_manifest = json.loads((base_run_dir / "manifest.json").read_text(encoding="utf-8-sig"))
base_results = [json.loads(x) for x in (base_run_dir / "responses.jsonl").read_text(encoding="utf-8-sig").splitlines() if x.strip()]
assert base_manifest["adapter"] is None, "Choose the original base-model run."
benchmark = base_manifest["benchmark"]
cases = benchmark["cases"]
prompts = base_manifest["messages_by_case"]
base_by_id = {r["id"]: r for r in base_results}
assert len(cases) == len(base_results) == len(base_by_id) == 24
assert set(base_by_id) == set(prompts) == {c["id"] for c in cases}
for case in cases:
    for field in ["user_message", "knowledge_context", "tool_result"]:
        assert base_by_id[case["id"]][field] == case[field], f"Base case differs: {case['id']}"
MODEL_ID = base_manifest["model_id"]
adapter_config = json.loads((adapter_dir / "adapter_config.json").read_text())
assert adapter_config["base_model_name_or_path"] == MODEL_ID, "Adapter belongs to a different base model."
MAX_NEW_TOKENS = base_manifest["generation_overrides"]["max_new_tokens"]
assert MAX_NEW_TOKENS == 128 and base_manifest["generation_overrides"]["do_sample"] is False
print("Project:", project_root)
print("Baseline:", base_run_dir.relative_to(project_root))
print("Adapter:", adapter_dir.relative_to(project_root))
print("Reusing all 24 original prompts; no reference answers are sent to the model.")

## Load V2 once
Use the same local GTX 1050 quantization settings as notebook 09. The original base tokenizer is intentionally retained to keep this comparison controlled; the saved base chat template is restored.
Adapter loading uses PEFT in inference mode ([documentation](https://huggingface.co/docs/peft/package_reference/peft_model)).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig
from peft import PeftModel

assert torch.cuda.is_available(), "CUDA-enabled PyTorch is required."
revision_args = {"revision": base_manifest["model_revision"]} if base_manifest.get("model_revision") else {}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **revision_args)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.chat_template = base_manifest["chat_template"]
assert tokenizer.pad_token_id == base_manifest["generation_overrides"]["pad_token_id"]

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    **revision_args,
    quantization_config=quantization,
    device_map={"": 0},
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=False)
model.generation_config = GenerationConfig.from_dict(base_manifest["generation_config"])
model.eval()
model.config.use_cache = True
print("Loaded V2 adapter on:", torch.cuda.get_device_name(0))

In [ ]:
def generate_case(messages):
    torch.cuda.synchronize()
    started = perf_counter()
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda:0")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    torch.cuda.synchronize()
    elapsed = perf_counter() - started
    new_ids = output[0, inputs["input_ids"].shape[1]:].tolist()

    eos = model.generation_config.eos_token_id
    if eos is None:
        eos = tokenizer.eos_token_id
    eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos])
    ended_with_eos = bool(new_ids) and new_ids[-1] in eos_ids

    return {
        "answer": tokenizer.decode(
            new_ids, skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip(),
        "input_tokens": inputs["input_ids"].shape[1],
        "generated_tokens": len(new_ids),
        "generation_seconds": round(elapsed, 3),
        "ended_with_eos": ended_with_eos,
        "hit_token_limit": len(new_ids) >= MAX_NEW_TOKENS and not ended_with_eos,
    }

## Save benchmark results alongside the V2 training history
Results are stored in `experiments/grounding_v2/<V2_TRAINING_RUN>/benchmark/`.

For the existing run, move the entire folder `experiments/grounding/v2_20261025T010214_503563Z` here and rename it `benchmark`. Keep its `manifest.json`, `responses.jsonl`, and `comparison.json` together. Do not move files while generation is running.

This cell reuses an existing benchmark folder after checking its configuration. Completed answers are skipped by the next cell. If the benchmark is already complete, no inference rerun is needed just because files moved.

For a deliberately different benchmark configuration, archive the existing benchmark folder under a descriptive name first.

In [ ]:
run_dir = v2_run_dir / "benchmark"
RESUME_EXISTING = run_dir.exists()
if RESUME_EXISTING:
    existing_manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
    run_id = existing_manifest["run_id"]
else:
    run_id = "v2_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

adapter_hashes = {name: sha256(adapter_dir / name) for name in ["adapter_config.json", "adapter_model.safetensors"]}
manifest = {
    **base_manifest,
    "run_id": run_id,
    "adapter": str(adapter_dir.relative_to(project_root)),
    "adapter_sha256": adapter_hashes,
    "baseline_run": str(base_run_dir.relative_to(project_root)),
    "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
    "packages": {p: metadata.version(p) for p in ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]},
    "quantization": quantization.to_dict(),
}
if RESUME_EXISTING:
    saved = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
    for key in ["adapter_sha256", "messages_by_case", "generation_overrides", "generation_config", "chat_template", "baseline_run", "packages"]:
        assert saved[key] == manifest[key], f"Run configuration changed: {key}. Start a new run."
else:
    run_dir.mkdir(parents=True, exist_ok=False)
    (run_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("V2 run folder:", run_dir.relative_to(project_root))

In [ ]:
results_path = run_dir / "responses.jsonl"
saved_manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
assert saved_manifest["messages_by_case"] == prompts, "Prompt changed; start a new run."
assert saved_manifest["generation_overrides"]["max_new_tokens"] == MAX_NEW_TOKENS
assert saved_manifest["chat_template"] == tokenizer.chat_template

done = set()
if results_path.exists():
    for line in results_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            done.add(json.loads(line)["id"])

with results_path.open("a", encoding="utf-8") as output_file:
    for index, case in enumerate(cases, start=1):
        if case["id"] in done:
            print(f"[{index}/{len(cases)}] {case['id']}: already saved")
            continue
        generated = generate_case(prompts[case["id"]])
        record = {
            **case,
            "model_label": "v2_epoch3",
            **generated,
            "review": {
                "correctness_0_to_2": None,
                "unsupported_claims_count": None,
                "unnecessary_apology": None,
                "unnecessarily_verbose": None,
                "notes": "",
            },
        }
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        output_file.flush()
        print(f"[{index}/{len(cases)}] {case['id']} / {case['prompt_type']} "
              f"— {generated['generation_seconds']:.1f}s "
              f"— token limit: {generated['hit_token_limit']}")
        print(generated["answer"], "\n")

print("Responses saved:", results_path.relative_to(project_root))

## Compare saved answers
These tables support semantic review, not automatic correctness grading. A paraphrase can be fully correct. Check required behaviour, unsupported claims, useful clarification, and truncation. Pay particular attention to G11, G21 and G24, where the original base run had difficulties.

The comparison file keeps all scoring criteria, supplied context, and both answers together. Successful execution alone does not confirm answer quality.

In [ ]:
from collections import Counter
from html import escape
from IPython.display import display, HTML

results = [json.loads(x) for x in results_path.read_text(encoding="utf-8").splitlines() if x.strip()]
v2_by_id = {r["id"]: r for r in results}
assert len(results) == len(v2_by_id) == 24, "Finish all 24 cases before reviewing."
comparison = []
for case in cases:
    case_id = case["id"]
    b, v = base_by_id[case_id], v2_by_id[case_id]
    item = {**case, "base_answer": b["answer"], "v2_answer": v["answer"],
            "base_hit_token_limit": b["hit_token_limit"], "v2_hit_token_limit": v["hit_token_limit"],
            "base_generation_seconds": b["generation_seconds"], "v2_generation_seconds": v["generation_seconds"],
            "review": {"base_correctness_0_to_2": None, "v2_correctness_0_to_2": None,
                       "v2_unsupported_claims_count": None, "regression": None, "notes": ""}}
    comparison.append(item)
    context = json.dumps({"knowledge": case["knowledge_context"], "tool_result": case["tool_result"]}, ensure_ascii=False, indent=2)
    display(HTML(f"<h3>{escape(case_id)} — {escape(case['prompt_type'])}</h3>"
                 + f"<p>{escape(case['user_message'])}</p><details><summary>Supplied context</summary><pre style='white-space:pre-wrap'>{escape(context)}</pre></details>"
                 + "<table style='width:100%;table-layout:fixed'><tr><th>Base</th><th>V2</th></tr><tr>"
                 + f"<td style='padding:10px;vertical-align:top'>{escape(b['answer'])}</td>"
                 + f"<td style='padding:10px;vertical-align:top'>{escape(v['answer'])}</td></tr></table>"
                 + "<p><b>Required:</b> " + escape("; ".join(case["required_behaviour"])) + "</p>"
                 + "<p><b>Forbidden:</b> " + escape("; ".join(case["forbidden_claims"])) + "</p>"))
comparison_path = run_dir / "comparison.json"
comparison_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8")
print("Completed:", len(results), "/ 24")
print("By prompt type:", dict(Counter(r["prompt_type"] for r in results)))
print("V2 token-limit hits:", sum(r["hit_token_limit"] for r in results))
print("V2 generation seconds:", round(sum(r["generation_seconds"] for r in results), 1))
print("Share these files:")
print(results_path)
print(comparison_path)

## Move to integration
After reviewing the 24 answers against the saved base results, use `models/adapter_v2` in Agent1's response generator if there are no material regressions. The next integration work is to connect the existing router/action executor and knowledge retrieval to `build_response_messages`, then pass those messages to the V2 response generator. We will preserve your `src/agent1` module layout.

This notebook measures response generation with supplied context. End-to-end integration will additionally check whether Agent1 retrieves the right context and executes the right action.